<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/model_evaluation/numerical_answer_evaluation_using_sentence_transformers_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Numerical Answer Evaluation using Sentence Transformers

This notebook demonstrates how to evaluate LLM-generated answers containing numerical values using **Bi-Encoders** and **Cross-Encoders** from Sentence Transformers.

## Approach Overview:

### Bi-Encoder (Sentence Transformer)
- Encodes text segments containing numbers into dense vectors
- Uses cosine similarity to find semantically similar number-context pairs
- Fast and efficient for candidate retrieval

### Cross-Encoder
- Takes pairs of texts and directly outputs a similarity/relevance score
- More accurate than bi-encoders for final matching
- Slower but better at capturing nuanced relationships

## Pipeline:
1. **Extract numbers** with surrounding context from both texts
2. **Bi-encoder retrieval**: Find candidate matches using embedding similarity
3. **Cross-encoder re-ranking**: Refine matches with pairwise scoring
4. **Statistical comparison**: Compare matched number values
5. **Decision**: Accept/Reject based on numerical accuracy


## 1. Installation and Setup


In [ ]:
# Install required packages
!pip install -q sentence-transformers pandas numpy matplotlib seaborn plotly scipy torch


In [ ]:
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Sentence Transformers
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Statistics
from scipy import stats
import torch

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✅ All imports successful!")


## 2. Load Sentence Transformer Models


In [ ]:
# Load Bi-Encoder and Cross-Encoder models
print("📥 Loading Sentence Transformer models...")

# Bi-Encoder for fast similarity search
# all-MiniLM-L6-v2 is fast and good for general semantic similarity
BI_ENCODER_MODEL = "all-MiniLM-L6-v2"
bi_encoder = SentenceTransformer(BI_ENCODER_MODEL)
print(f"✅ Bi-Encoder loaded: {BI_ENCODER_MODEL}")

# Cross-Encoder for accurate pairwise scoring
# cross-encoder/ms-marco-MiniLM-L-6-v2 is good for relevance/similarity
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f"✅ Cross-Encoder loaded: {CROSS_ENCODER_MODEL}")

print("\n📊 Model Info:")
print(f"   Bi-Encoder embedding dimension: {bi_encoder.get_sentence_embedding_dimension()}")


## 3. Number Extraction with Context

Extract numbers from text along with their surrounding context for semantic matching.


In [ ]:
@dataclass
class NumberWithContext:
    """A number extracted from text with its surrounding context."""
    value: float
    original_text: str
    context_sentence: str  # Full sentence or context window
    context_before: str
    context_after: str
    start_pos: int
    end_pos: int
    number_type: str
    embedding: Optional[np.ndarray] = None
    
    def get_context_text(self) -> str:
        """Get the text to use for embedding."""
        return self.context_sentence
    
    def __repr__(self):
        return f"NumberWithContext({self.value}, context='{self.context_sentence[:50]}...')"


class NumberContextExtractor:
    """
    Extracts numbers from text with rich context for semantic matching.
    """
    
    def __init__(self, context_window: int = 50):
        self.context_window = context_window
        
        # Regex patterns for different number formats
        self.patterns = {
            'scientific': r'[-+]?\d+\.?\d*[eE][-+]?\d+',
            'percentage': r'[-+]?\d+\.?\d*\s*%',
            'currency': r'[$€£¥₹]\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*(?:dollars?|euros?|pounds?|billion|million|trillion)',
            'fraction': r'\d+\s*/\s*\d+',
            'decimal_comma': r'[-+]?\d{1,3}(?:,\d{3})*\.\d+',
            'integer_comma': r'[-+]?\d{1,3}(?:,\d{3})+(?!\.\d)',
            'decimal': r'[-+]?\d+\.\d+',
            'integer': r'[-+]?\d+',
        }
        
        self.compiled_patterns = {
            name: re.compile(pattern) 
            for name, pattern in self.patterns.items()
        }
    
    def _find_sentence_boundary(self, text: str, pos: int, direction: str) -> int:
        """Find the nearest sentence boundary."""
        sentence_enders = '.!?'
        
        if direction == 'before':
            # Search backwards for sentence end
            for i in range(pos - 1, max(0, pos - 200), -1):
                if text[i] in sentence_enders:
                    return i + 1
            return max(0, pos - self.context_window)
        else:
            # Search forwards for sentence end
            for i in range(pos, min(len(text), pos + 200)):
                if text[i] in sentence_enders:
                    return i + 1
            return min(len(text), pos + self.context_window)
    
    def _parse_number(self, text: str, number_type: str) -> float:
        """Convert extracted text to float value."""
        text = text.strip().lower()
        
        # Handle multipliers
        multiplier = 1
        if 'trillion' in text:
            multiplier = 1e12
            text = re.sub(r'\s*trillion\s*', '', text)
        elif 'billion' in text:
            multiplier = 1e9
            text = re.sub(r'\s*billion\s*', '', text)
        elif 'million' in text:
            multiplier = 1e6
            text = re.sub(r'\s*million\s*', '', text)
        
        if number_type == 'percentage':
            return float(text.replace('%', '').replace(',', '').strip())
        elif number_type == 'currency':
            cleaned = re.sub(r'[$€£¥₹]|\b(?:dollars?|euros?|pounds?)\b', '', text)
            return float(cleaned.replace(',', '').strip()) * multiplier
        elif number_type == 'fraction':
            parts = text.split('/')
            return float(parts[0]) / float(parts[1])
        elif number_type in ['decimal_comma', 'integer_comma']:
            return float(text.replace(',', '')) * multiplier
        else:
            return float(text.replace(',', '')) * multiplier
    
    def extract(self, text: str) -> List[NumberWithContext]:
        """Extract all numbers with their context."""
        extracted = []
        used_positions = set()
        
        for number_type in ['scientific', 'percentage', 'currency', 'fraction', 
                           'decimal_comma', 'integer_comma', 'decimal', 'integer']:
            pattern = self.compiled_patterns[number_type]
            
            for match in pattern.finditer(text):
                start, end = match.start(), match.end()
                
                if any(start < used_end and end > used_start 
                       for used_start, used_end in used_positions):
                    continue
                
                original_text = match.group()
                
                try:
                    value = self._parse_number(original_text, number_type)
                    
                    # Get context window
                    context_start = max(0, start - self.context_window)
                    context_end = min(len(text), end + self.context_window)
                    
                    # Try to find sentence boundaries for better context
                    sent_start = self._find_sentence_boundary(text, start, 'before')
                    sent_end = self._find_sentence_boundary(text, end, 'after')
                    
                    context_sentence = text[sent_start:sent_end].strip()
                    context_before = text[context_start:start].strip()
                    context_after = text[end:context_end].strip()
                    
                    extracted.append(NumberWithContext(
                        value=value,
                        original_text=original_text,
                        context_sentence=context_sentence,
                        context_before=context_before,
                        context_after=context_after,
                        start_pos=start,
                        end_pos=end,
                        number_type=number_type
                    ))
                    
                    used_positions.add((start, end))
                    
                except (ValueError, ZeroDivisionError):
                    continue
        
        extracted.sort(key=lambda x: x.start_pos)
        return extracted

# Test the extractor
extractor = NumberContextExtractor()

test_text = """
The company reported revenue of $2.5 billion in Q3 2023, representing a 15% increase 
from Q2. The profit margin improved to 23.4%, up from 21.8% in the previous quarter.
Total employees: 45,000. Stock price reached $142.50 per share.
"""

numbers = extractor.extract(test_text)

print("📊 Number Extraction with Context")
print("="*70)
print(f"\nExtracted {len(numbers)} numbers:")
for num in numbers:
    print(f"\n  Value: {num.value:,.4g} ({num.number_type})")
    print(f"  Original: '{num.original_text}'")
    print(f"  Context: '{num.context_sentence[:80]}...'")


## 4. Semantic Number Matcher using Bi-Encoder and Cross-Encoder

This module matches numbers from the model answer to ground truth using:
1. **Bi-Encoder**: Fast semantic similarity for candidate retrieval
2. **Cross-Encoder**: Accurate re-ranking for final matching


In [ ]:
@dataclass
class SemanticNumberMatch:
    """A matched pair of numbers with semantic similarity scores."""
    model_number: NumberWithContext
    truth_number: NumberWithContext
    bi_encoder_score: float  # Cosine similarity from bi-encoder
    cross_encoder_score: float  # Direct score from cross-encoder
    combined_score: float  # Weighted combination
    match_confidence: str  # 'high', 'medium', 'low'


class SemanticNumberMatcher:
    """
    Matches numbers from model answer to ground truth using Sentence Transformers.
    
    Two-stage approach:
    1. Bi-Encoder: Encode all contexts, find candidates via cosine similarity
    2. Cross-Encoder: Re-rank candidates with pairwise scoring
    """
    
    def __init__(
        self, 
        bi_encoder: SentenceTransformer,
        cross_encoder: CrossEncoder,
        bi_encoder_weight: float = 0.3,
        cross_encoder_weight: float = 0.7,
        similarity_threshold: float = 0.5,
        top_k_candidates: int = 3
    ):
        self.bi_encoder = bi_encoder
        self.cross_encoder = cross_encoder
        self.bi_encoder_weight = bi_encoder_weight
        self.cross_encoder_weight = cross_encoder_weight
        self.similarity_threshold = similarity_threshold
        self.top_k_candidates = top_k_candidates
    
    def _compute_embeddings(self, numbers: List[NumberWithContext]) -> np.ndarray:
        """Compute bi-encoder embeddings for number contexts."""
        if not numbers:
            return np.array([])
        
        texts = [num.get_context_text() for num in numbers]
        embeddings = self.bi_encoder.encode(texts, convert_to_numpy=True)
        
        # Store embeddings in the number objects
        for num, emb in zip(numbers, embeddings):
            num.embedding = emb
        
        return embeddings
    
    def _compute_cross_encoder_scores(
        self, 
        model_numbers: List[NumberWithContext],
        truth_numbers: List[NumberWithContext],
        candidate_pairs: List[Tuple[int, int]]
    ) -> Dict[Tuple[int, int], float]:
        """Compute cross-encoder scores for candidate pairs."""
        if not candidate_pairs:
            return {}
        
        # Prepare pairs for cross-encoder
        pairs = []
        for model_idx, truth_idx in candidate_pairs:
            model_text = model_numbers[model_idx].get_context_text()
            truth_text = truth_numbers[truth_idx].get_context_text()
            pairs.append([model_text, truth_text])
        
        # Get cross-encoder scores
        scores = self.cross_encoder.predict(pairs)
        
        # Normalize scores to [0, 1] range using sigmoid
        normalized_scores = 1 / (1 + np.exp(-scores))
        
        return {
            pair: float(score) 
            for pair, score in zip(candidate_pairs, normalized_scores)
        }
    
    def _get_confidence_level(self, combined_score: float) -> str:
        """Determine match confidence based on combined score."""
        if combined_score >= 0.8:
            return 'high'
        elif combined_score >= 0.6:
            return 'medium'
        else:
            return 'low'
    
    def match(
        self,
        model_numbers: List[NumberWithContext],
        truth_numbers: List[NumberWithContext]
    ) -> List[SemanticNumberMatch]:
        """
        Match numbers from model answer to ground truth using semantic similarity.
        
        Returns:
            List of SemanticNumberMatch objects representing matched pairs.
        """
        if not model_numbers or not truth_numbers:
            return []
        
        # Stage 1: Compute bi-encoder embeddings
        model_embeddings = self._compute_embeddings(model_numbers)
        truth_embeddings = self._compute_embeddings(truth_numbers)
        
        # Compute similarity matrix
        similarity_matrix = cosine_similarity(model_embeddings, truth_embeddings)
        
        # Find candidate pairs (top-k for each model number)
        candidate_pairs = []
        for model_idx in range(len(model_numbers)):
            top_k_indices = np.argsort(similarity_matrix[model_idx])[-self.top_k_candidates:][::-1]
            for truth_idx in top_k_indices:
                if similarity_matrix[model_idx, truth_idx] >= self.similarity_threshold * 0.5:
                    candidate_pairs.append((model_idx, int(truth_idx)))
        
        # Stage 2: Cross-encoder re-ranking
        cross_encoder_scores = self._compute_cross_encoder_scores(
            model_numbers, truth_numbers, candidate_pairs
        )
        
        # Compute combined scores and select best matches
        pair_scores = {}
        for model_idx, truth_idx in candidate_pairs:
            bi_score = similarity_matrix[model_idx, truth_idx]
            ce_score = cross_encoder_scores.get((model_idx, truth_idx), 0.0)
            combined = (
                self.bi_encoder_weight * bi_score + 
                self.cross_encoder_weight * ce_score
            )
            pair_scores[(model_idx, truth_idx)] = {
                'bi_score': bi_score,
                'ce_score': ce_score,
                'combined': combined
            }
        
        # Greedy matching: assign each model number to best available truth number
        matches = []
        used_truth_indices = set()
        
        # Sort pairs by combined score (descending)
        sorted_pairs = sorted(
            pair_scores.items(), 
            key=lambda x: x[1]['combined'], 
            reverse=True
        )
        
        for (model_idx, truth_idx), scores in sorted_pairs:
            if truth_idx in used_truth_indices:
                continue
            if scores['combined'] < self.similarity_threshold:
                continue
            
            # Check if model number already matched
            if any(m.model_number is model_numbers[model_idx] for m in matches):
                continue
            
            match = SemanticNumberMatch(
                model_number=model_numbers[model_idx],
                truth_number=truth_numbers[truth_idx],
                bi_encoder_score=float(scores['bi_score']),
                cross_encoder_score=scores['ce_score'],
                combined_score=scores['combined'],
                match_confidence=self._get_confidence_level(scores['combined'])
            )
            matches.append(match)
            used_truth_indices.add(truth_idx)
        
        return matches


# Initialize the semantic matcher
matcher = SemanticNumberMatcher(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    bi_encoder_weight=0.3,
    cross_encoder_weight=0.7,
    similarity_threshold=0.5
)

print("✅ SemanticNumberMatcher initialized")


## 5. Numerical Comparison and Evaluation

Statistical comparison of matched number pairs with configurable tolerances.


In [ ]:
class AcceptanceDecision(Enum):
    """Evaluation decision for numerical accuracy."""
    ACCEPT = "accept"
    MARGINAL = "marginal"
    REJECT = "reject"


@dataclass
class NumberPairComparison:
    """Detailed comparison result for a matched number pair."""
    match: SemanticNumberMatch
    model_value: float
    truth_value: float
    absolute_error: float
    relative_error: float
    percentage_error: float
    order_of_magnitude_diff: float
    is_within_tolerance: bool
    decision: AcceptanceDecision


@dataclass
class EvaluationResult:
    """Complete evaluation result for a model answer."""
    question: str
    model_answer: str
    ground_truth: str
    
    # Numbers found
    model_numbers: List[NumberWithContext]
    truth_numbers: List[NumberWithContext]
    
    # Matches and comparisons
    matches: List[SemanticNumberMatch]
    comparisons: List[NumberPairComparison]
    
    # Summary statistics
    total_model_numbers: int
    total_truth_numbers: int
    matched_pairs: int
    within_tolerance_count: int
    
    # Scores
    matching_coverage: float
    numerical_accuracy_score: float
    semantic_match_score: float
    overall_score: float
    
    # Decision
    overall_decision: AcceptanceDecision


class SentenceTransformerNumericalEvaluator:
    """
    Evaluator that uses Sentence Transformers to match and compare
    numerical values between model answers and ground truth.
    """
    
    def __init__(
        self,
        bi_encoder: SentenceTransformer,
        cross_encoder: CrossEncoder,
        relative_tolerance: float = 0.10,
        absolute_tolerance: float = 0.01,
        order_of_magnitude_tolerance: float = 0.5,
        marginal_multiplier: float = 2.0,
        min_semantic_score: float = 0.5,
        accept_threshold: float = 0.7,
        marginal_threshold: float = 0.5
    ):
        self.extractor = NumberContextExtractor()
        self.matcher = SemanticNumberMatcher(
            bi_encoder=bi_encoder,
            cross_encoder=cross_encoder,
            similarity_threshold=min_semantic_score
        )
        
        # Numerical tolerances
        self.relative_tolerance = relative_tolerance
        self.absolute_tolerance = absolute_tolerance
        self.order_of_magnitude_tolerance = order_of_magnitude_tolerance
        self.marginal_multiplier = marginal_multiplier
        
        # Decision thresholds
        self.accept_threshold = accept_threshold
        self.marginal_threshold = marginal_threshold
    
    def _compare_pair(self, match: SemanticNumberMatch) -> NumberPairComparison:
        """Compare numerical values in a matched pair."""
        model_val = match.model_number.value
        truth_val = match.truth_number.value
        
        # Calculate errors
        absolute_error = abs(model_val - truth_val)
        
        if truth_val != 0:
            relative_error = abs(model_val - truth_val) / abs(truth_val)
        else:
            relative_error = float('inf') if model_val != 0 else 0.0
        
        percentage_error = relative_error * 100
        
        # Order of magnitude difference
        if model_val > 0 and truth_val > 0:
            order_of_magnitude_diff = abs(np.log10(model_val) - np.log10(truth_val))
        elif model_val == 0 and truth_val == 0:
            order_of_magnitude_diff = 0.0
        else:
            order_of_magnitude_diff = float('inf')
        
        # Check tolerance
        within_tolerance = (
            relative_error <= self.relative_tolerance or
            absolute_error <= self.absolute_tolerance
        ) and order_of_magnitude_diff <= self.order_of_magnitude_tolerance
        
        # Determine decision
        if within_tolerance:
            decision = AcceptanceDecision.ACCEPT
        elif (relative_error <= self.relative_tolerance * self.marginal_multiplier and
              order_of_magnitude_diff <= self.order_of_magnitude_tolerance * 1.5):
            decision = AcceptanceDecision.MARGINAL
        else:
            decision = AcceptanceDecision.REJECT
        
        return NumberPairComparison(
            match=match,
            model_value=model_val,
            truth_value=truth_val,
            absolute_error=absolute_error,
            relative_error=relative_error,
            percentage_error=percentage_error,
            order_of_magnitude_diff=order_of_magnitude_diff,
            is_within_tolerance=within_tolerance,
            decision=decision
        )
    
    def evaluate(
        self,
        question: str,
        model_answer: str,
        ground_truth: str
    ) -> EvaluationResult:
        """
        Evaluate a model answer against ground truth.
        
        Returns:
            EvaluationResult with detailed analysis and decision.
        """
        # Extract numbers with context
        model_numbers = self.extractor.extract(model_answer)
        truth_numbers = self.extractor.extract(ground_truth)
        
        # Semantic matching
        matches = self.matcher.match(model_numbers, truth_numbers)
        
        # Compare matched pairs
        comparisons = [self._compare_pair(m) for m in matches]
        
        # Calculate statistics
        total_model = len(model_numbers)
        total_truth = len(truth_numbers)
        matched_pairs = len(matches)
        within_tolerance = sum(1 for c in comparisons if c.is_within_tolerance)
        
        # Calculate scores
        matching_coverage = (
            matched_pairs / total_truth if total_truth > 0 else 
            (1.0 if total_model == 0 else 0.0)
        )
        
        numerical_accuracy = (
            within_tolerance / matched_pairs if matched_pairs > 0 else 
            (1.0 if total_truth == 0 else 0.0)
        )
        
        semantic_score = (
            np.mean([m.combined_score for m in matches]) if matches else 0.0
        )
        
        # Overall score (weighted combination)
        overall_score = (
            0.4 * matching_coverage +
            0.4 * numerical_accuracy +
            0.2 * semantic_score
        )
        
        # Overall decision
        if overall_score >= self.accept_threshold and numerical_accuracy >= 0.7:
            overall_decision = AcceptanceDecision.ACCEPT
        elif overall_score >= self.marginal_threshold:
            overall_decision = AcceptanceDecision.MARGINAL
        else:
            overall_decision = AcceptanceDecision.REJECT
        
        return EvaluationResult(
            question=question,
            model_answer=model_answer,
            ground_truth=ground_truth,
            model_numbers=model_numbers,
            truth_numbers=truth_numbers,
            matches=matches,
            comparisons=comparisons,
            total_model_numbers=total_model,
            total_truth_numbers=total_truth,
            matched_pairs=matched_pairs,
            within_tolerance_count=within_tolerance,
            matching_coverage=matching_coverage,
            numerical_accuracy_score=numerical_accuracy,
            semantic_match_score=semantic_score,
            overall_score=overall_score,
            overall_decision=overall_decision
        )


# Initialize the evaluator
evaluator = SentenceTransformerNumericalEvaluator(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    relative_tolerance=0.10,  # 10% relative error allowed
    absolute_tolerance=0.01,  # For small numbers
    order_of_magnitude_tolerance=0.5
)

print("✅ SentenceTransformerNumericalEvaluator initialized")


## 6. Sample Evaluation Examples

Let's test the evaluator with various example scenarios.


In [ ]:
# Sample evaluation examples
examples = [
    {
        "id": 1,
        "name": "Financial Report - Accurate",
        "question": "What were Apple's Q3 2023 financial results?",
        "model_answer": """Apple reported revenue of $81.8 billion for Q3 2023, representing a 
        1.4% year-over-year decline. The gross margin was 44.5%, while operating income 
        reached $23.1 billion. Earnings per share came in at $1.26.""",
        "ground_truth": """Apple's Q3 2023 revenue was $81.8 billion, down 1.4% from the 
        prior year. Gross margin stood at 44.5%, operating income was $23.1 billion, 
        and EPS was $1.26."""
    },
    {
        "id": 2,
        "name": "Scientific Data - Minor Error",
        "question": "What is the speed of light and Earth's distance from the Sun?",
        "model_answer": """The speed of light in a vacuum is approximately 299,792 kilometers 
        per second. Earth is located at an average distance of 150 million kilometers from 
        the Sun, also known as 1 AU (astronomical unit).""",
        "ground_truth": """Light travels at exactly 299,792.458 kilometers per second in a 
        vacuum. The Earth orbits the Sun at a mean distance of 149.6 million kilometers, 
        which defines 1 astronomical unit (AU)."""
    },
    {
        "id": 3,
        "name": "Population Statistics - Significant Error",
        "question": "What is the population of Tokyo and New York City?",
        "model_answer": """Tokyo has a population of approximately 14.5 million people in 
        the city proper, making it one of the largest cities in the world. New York City 
        has around 9 million residents.""",
        "ground_truth": """The city proper of Tokyo has about 13.96 million inhabitants. 
        New York City's population is approximately 8.34 million people."""
    },
    {
        "id": 4,
        "name": "Economic Data - Order of Magnitude Error",
        "question": "What is the US GDP and national debt?",
        "model_answer": """The US GDP is approximately 25 trillion dollars. The national 
        debt stands at around 340 billion dollars as of 2023.""",
        "ground_truth": """The United States has a GDP of about $25.5 trillion. The 
        national debt has reached approximately $34 trillion."""
    },
    {
        "id": 5,
        "name": "Weather Data - Mixed Accuracy",
        "question": "What were the temperature records for Death Valley?",
        "model_answer": """Death Valley holds the record for the highest reliably recorded 
        air temperature on Earth at 134°F (56.7°C), set on July 10, 1913. The average 
        summer high is about 115°F, and the record low is 15°F.""",
        "ground_truth": """The highest temperature ever recorded was 134°F (56.7°C) in 
        Death Valley on July 10, 1913. Summer highs average around 116°F, and the 
        record low temperature was 15°F (-9.4°C)."""
    },
]

print(f"📝 Loaded {len(examples)} evaluation examples")


In [ ]:
# Run evaluation on all examples
results = []

print("🔍 Running Evaluation with Sentence Transformers")
print("="*80)

for ex in examples:
    result = evaluator.evaluate(
        question=ex["question"],
        model_answer=ex["model_answer"],
        ground_truth=ex["ground_truth"]
    )
    results.append({
        "example": ex,
        "result": result
    })
    
    # Print summary
    decision_emoji = {
        AcceptanceDecision.ACCEPT: "✅",
        AcceptanceDecision.MARGINAL: "⚠️",
        AcceptanceDecision.REJECT: "❌"
    }
    
    print(f"\n📊 Example {ex['id']}: {ex['name']}")
    print("-" * 60)
    print(f"   Decision: {decision_emoji[result.overall_decision]} {result.overall_decision.value.upper()}")
    print(f"   Numbers: {result.total_model_numbers} model / {result.total_truth_numbers} truth")
    print(f"   Matched: {result.matched_pairs} pairs")
    print(f"   Within Tolerance: {result.within_tolerance_count}/{result.matched_pairs}")
    print(f"   Scores:")
    print(f"      - Matching Coverage:    {result.matching_coverage:.2%}")
    print(f"      - Numerical Accuracy:   {result.numerical_accuracy_score:.2%}")
    print(f"      - Semantic Match Score: {result.semantic_match_score:.3f}")
    print(f"      - Overall Score:        {result.overall_score:.3f}")


## 7. Detailed Match Analysis

Examine the semantic matching results in detail.


In [ ]:
def display_detailed_matches(result: EvaluationResult, example_name: str):
    """Display detailed match information with semantic scores."""
    print(f"\n{'='*80}")
    print(f"🔎 Detailed Matches: {example_name}")
    print(f"{'='*80}")
    
    if not result.matches:
        print("   No matches found!")
        return
    
    for i, (match, comp) in enumerate(zip(result.matches, result.comparisons), 1):
        decision_emoji = {
            AcceptanceDecision.ACCEPT: "✅",
            AcceptanceDecision.MARGINAL: "⚠️",
            AcceptanceDecision.REJECT: "❌"
        }
        
        print(f"\n   Match {i}: {decision_emoji[comp.decision]}")
        print(f"   {'-'*60}")
        print(f"   Model Value:  {comp.model_value:,.4g}")
        print(f"   Truth Value:  {comp.truth_value:,.4g}")
        print(f"   ")
        print(f"   📊 Semantic Scores:")
        print(f"      Bi-Encoder:    {match.bi_encoder_score:.3f}")
        print(f"      Cross-Encoder: {match.cross_encoder_score:.3f}")
        print(f"      Combined:      {match.combined_score:.3f} ({match.match_confidence} confidence)")
        print(f"   ")
        print(f"   📏 Numerical Metrics:")
        print(f"      Absolute Error:  {comp.absolute_error:,.4g}")
        print(f"      Relative Error:  {comp.relative_error:.2%}")
        print(f"      Order Magnitude: {comp.order_of_magnitude_diff:.3f}")
        print(f"   ")
        print(f"   📝 Context:")
        print(f"      Model: '{match.model_number.context_sentence[:70]}...'")
        print(f"      Truth: '{match.truth_number.context_sentence[:70]}...'")

# Show detailed matches for a few examples
display_detailed_matches(results[0]["result"], results[0]["example"]["name"])
display_detailed_matches(results[3]["result"], results[3]["example"]["name"])


## 8. Visualizations

Visualize the evaluation results and semantic matching scores.


In [ ]:
# Create summary DataFrame
summary_data = []
for r in results:
    ex = r["example"]
    res = r["result"]
    summary_data.append({
        "Example": ex["name"],
        "Decision": res.overall_decision.value,
        "Model Numbers": res.total_model_numbers,
        "Truth Numbers": res.total_truth_numbers,
        "Matched": res.matched_pairs,
        "Within Tolerance": res.within_tolerance_count,
        "Coverage": res.matching_coverage,
        "Numerical Accuracy": res.numerical_accuracy_score,
        "Semantic Score": res.semantic_match_score,
        "Overall Score": res.overall_score
    })

summary_df = pd.DataFrame(summary_data)

# Display summary table
print("📊 Evaluation Summary")
print("="*100)
display(summary_df)


In [ ]:
# Visualization 1: Overall Scores by Example
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Decision colors
decision_colors = {
    'accept': '#2ecc71',
    'marginal': '#f39c12', 
    'reject': '#e74c3c'
}

# Plot 1: Overall Scores
ax1 = axes[0, 0]
bars = ax1.barh(summary_df['Example'], summary_df['Overall Score'])
for bar, decision in zip(bars, summary_df['Decision']):
    bar.set_color(decision_colors[decision])
ax1.axvline(x=0.7, color='green', linestyle='--', alpha=0.7, label='Accept threshold')
ax1.axvline(x=0.5, color='orange', linestyle='--', alpha=0.7, label='Marginal threshold')
ax1.set_xlabel('Overall Score')
ax1.set_title('Overall Evaluation Scores')
ax1.legend()
ax1.set_xlim(0, 1)

# Plot 2: Score Components
ax2 = axes[0, 1]
x = range(len(summary_df))
width = 0.25
ax2.bar([i - width for i in x], summary_df['Coverage'], width, label='Coverage', color='#3498db')
ax2.bar(x, summary_df['Numerical Accuracy'], width, label='Numerical Accuracy', color='#9b59b6')
ax2.bar([i + width for i in x], summary_df['Semantic Score'], width, label='Semantic Score', color='#1abc9c')
ax2.set_xticks(x)
ax2.set_xticklabels([ex['name'][:15] + '...' for ex in examples], rotation=45, ha='right')
ax2.set_ylabel('Score')
ax2.set_title('Score Components by Example')
ax2.legend()
ax2.set_ylim(0, 1.1)

# Plot 3: Matching Statistics
ax3 = axes[1, 0]
x = range(len(summary_df))
width = 0.35
ax3.bar([i - width/2 for i in x], summary_df['Truth Numbers'], width, label='Ground Truth Numbers', color='#34495e')
ax3.bar([i + width/2 for i in x], summary_df['Matched'], width, label='Matched Pairs', color='#2ecc71')
ax3.set_xticks(x)
ax3.set_xticklabels([ex['name'][:15] + '...' for ex in examples], rotation=45, ha='right')
ax3.set_ylabel('Count')
ax3.set_title('Matching Coverage')
ax3.legend()

# Plot 4: Decision Distribution
ax4 = axes[1, 1]
decision_counts = summary_df['Decision'].value_counts()
colors_pie = [decision_colors.get(d, 'gray') for d in decision_counts.index]
wedges, texts, autotexts = ax4.pie(
    decision_counts.values, 
    labels=decision_counts.index, 
    autopct='%1.1f%%',
    colors=colors_pie,
    explode=[0.05] * len(decision_counts)
)
ax4.set_title('Decision Distribution')

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Bi-Encoder vs Cross-Encoder Scores
all_matches = []
for r in results:
    res = r["result"]
    ex_name = r["example"]["name"]
    for match, comp in zip(res.matches, res.comparisons):
        all_matches.append({
            'Example': ex_name[:20],
            'Bi-Encoder Score': match.bi_encoder_score,
            'Cross-Encoder Score': match.cross_encoder_score,
            'Combined Score': match.combined_score,
            'Relative Error': min(comp.relative_error, 1.0),  # Cap for visualization
            'Decision': comp.decision.value,
            'Model Value': comp.model_value,
            'Truth Value': comp.truth_value
        })

matches_df = pd.DataFrame(all_matches)

if not matches_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Scatter: Bi-Encoder vs Cross-Encoder
    ax1 = axes[0]
    decision_markers = {'accept': 'o', 'marginal': 's', 'reject': 'X'}
    for decision in matches_df['Decision'].unique():
        subset = matches_df[matches_df['Decision'] == decision]
        ax1.scatter(
            subset['Bi-Encoder Score'], 
            subset['Cross-Encoder Score'],
            c=decision_colors.get(decision, 'gray'),
            marker=decision_markers.get(decision, 'o'),
            s=100,
            label=decision.capitalize(),
            alpha=0.7
        )
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax1.set_xlabel('Bi-Encoder Score')
    ax1.set_ylabel('Cross-Encoder Score')
    ax1.set_title('Bi-Encoder vs Cross-Encoder Scores')
    ax1.legend()
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # Scatter: Combined Score vs Relative Error
    ax2 = axes[1]
    for decision in matches_df['Decision'].unique():
        subset = matches_df[matches_df['Decision'] == decision]
        ax2.scatter(
            subset['Combined Score'], 
            subset['Relative Error'] * 100,
            c=decision_colors.get(decision, 'gray'),
            marker=decision_markers.get(decision, 'o'),
            s=100,
            label=decision.capitalize(),
            alpha=0.7
        )
    ax2.axhline(y=10, color='green', linestyle='--', alpha=0.5, label='10% tolerance')
    ax2.axhline(y=20, color='orange', linestyle='--', alpha=0.5, label='20% marginal')
    ax2.set_xlabel('Combined Semantic Score')
    ax2.set_ylabel('Relative Error (%)')
    ax2.set_title('Semantic Score vs Numerical Error')
    ax2.legend()
    
    # Box plot: Scores by Decision
    ax3 = axes[2]
    score_melt = matches_df.melt(
        id_vars=['Decision'],
        value_vars=['Bi-Encoder Score', 'Cross-Encoder Score'],
        var_name='Score Type',
        value_name='Score'
    )
    colors = [decision_colors.get(d, 'gray') for d in ['accept', 'marginal', 'reject']]
    score_melt['Decision'] = pd.Categorical(score_melt['Decision'], categories=['accept', 'marginal', 'reject'])
    
    positions = {'accept': [1, 1.5], 'marginal': [2.5, 3], 'reject': [4, 4.5]}
    for decision in ['accept', 'marginal', 'reject']:
        subset = matches_df[matches_df['Decision'] == decision]
        if not subset.empty:
            bp1 = ax3.boxplot(subset['Bi-Encoder Score'].dropna(), positions=[positions[decision][0]], 
                           widths=0.4, patch_artist=True)
            bp2 = ax3.boxplot(subset['Cross-Encoder Score'].dropna(), positions=[positions[decision][1]], 
                           widths=0.4, patch_artist=True)
            bp1['boxes'][0].set_facecolor('#3498db')
            bp2['boxes'][0].set_facecolor('#e74c3c')
    
    ax3.set_xticks([1.25, 2.75, 4.25])
    ax3.set_xticklabels(['Accept', 'Marginal', 'Reject'])
    ax3.set_ylabel('Score')
    ax3.set_title('Score Distribution by Decision')
    ax3.legend([plt.Rectangle((0,0),1,1, fc='#3498db'), 
                plt.Rectangle((0,0),1,1, fc='#e74c3c')],
               ['Bi-Encoder', 'Cross-Encoder'], loc='lower left')
    
    plt.tight_layout()
    plt.show()
else:
    print("No matches to visualize")


In [ ]:
# Interactive Plotly Visualization
if not matches_df.empty:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Model vs Truth Values', 'Semantic Matching Analysis'),
        specs=[[{"type": "scatter"}, {"type": "scatter"}]]
    )
    
    # Left plot: Model vs Truth Values (log scale)
    for decision in matches_df['Decision'].unique():
        subset = matches_df[matches_df['Decision'] == decision]
        fig.add_trace(
            go.Scatter(
                x=subset['Truth Value'],
                y=subset['Model Value'],
                mode='markers',
                name=f'{decision.capitalize()}',
                marker=dict(
                    size=12,
                    color=decision_colors.get(decision, 'gray'),
                    opacity=0.7
                ),
                text=subset['Example'],
                hovertemplate="<b>%{text}</b><br>" +
                              "Truth: %{x:,.4g}<br>" +
                              "Model: %{y:,.4g}<br>" +
                              "<extra></extra>"
            ),
            row=1, col=1
        )
    
    # Add perfect match line
    max_val = max(matches_df['Truth Value'].max(), matches_df['Model Value'].max())
    min_val = min(matches_df['Truth Value'].min(), matches_df['Model Value'].min())
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            name='Perfect Match',
            line=dict(color='gray', dash='dash'),
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Right plot: Bi-Encoder vs Cross-Encoder with error as bubble size
    for decision in matches_df['Decision'].unique():
        subset = matches_df[matches_df['Decision'] == decision]
        fig.add_trace(
            go.Scatter(
                x=subset['Bi-Encoder Score'],
                y=subset['Cross-Encoder Score'],
                mode='markers',
                name=f'{decision.capitalize()} (semantic)',
                marker=dict(
                    size=np.clip(subset['Relative Error'] * 100 + 5, 10, 50),
                    color=decision_colors.get(decision, 'gray'),
                    opacity=0.7,
                    line=dict(width=1, color='white')
                ),
                text=subset.apply(lambda r: f"{r['Example']}<br>Error: {r['Relative Error']*100:.1f}%", axis=1),
                hovertemplate="<b>%{text}</b><br>" +
                              "Bi-Encoder: %{x:.3f}<br>" +
                              "Cross-Encoder: %{y:.3f}<br>" +
                              "<extra></extra>",
                showlegend=False
            ),
            row=1, col=2
        )
    
    fig.update_xaxes(title_text="Ground Truth Value", type="log", row=1, col=1)
    fig.update_yaxes(title_text="Model Value", type="log", row=1, col=1)
    fig.update_xaxes(title_text="Bi-Encoder Score", range=[0, 1], row=1, col=2)
    fig.update_yaxes(title_text="Cross-Encoder Score", range=[0, 1], row=1, col=2)
    
    fig.update_layout(
        height=500,
        title_text="<b>Numerical Evaluation with Sentence Transformers</b>",
        title_x=0.5,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )
    
    fig.show()


## 9. Comparing Bi-Encoder Only vs Bi-Encoder + Cross-Encoder

Let's compare the matching quality when using only the bi-encoder versus the combined approach.


In [ ]:
class BiEncoderOnlyMatcher(SemanticNumberMatcher):
    """Matcher using only bi-encoder (no cross-encoder re-ranking)."""
    
    def match(
        self,
        model_numbers: List[NumberWithContext],
        truth_numbers: List[NumberWithContext]
    ) -> List[SemanticNumberMatch]:
        """Match using bi-encoder similarity only."""
        if not model_numbers or not truth_numbers:
            return []
        
        # Compute bi-encoder embeddings
        model_embeddings = self._compute_embeddings(model_numbers)
        truth_embeddings = self._compute_embeddings(truth_numbers)
        
        # Compute similarity matrix
        similarity_matrix = cosine_similarity(model_embeddings, truth_embeddings)
        
        # Greedy matching based on bi-encoder scores only
        matches = []
        used_truth_indices = set()
        
        # Get all pairs with scores
        pair_scores = []
        for model_idx in range(len(model_numbers)):
            for truth_idx in range(len(truth_numbers)):
                score = similarity_matrix[model_idx, truth_idx]
                if score >= self.similarity_threshold:
                    pair_scores.append((model_idx, truth_idx, score))
        
        # Sort by score descending
        pair_scores.sort(key=lambda x: x[2], reverse=True)
        
        used_model_indices = set()
        for model_idx, truth_idx, bi_score in pair_scores:
            if model_idx in used_model_indices or truth_idx in used_truth_indices:
                continue
            
            match = SemanticNumberMatch(
                model_number=model_numbers[model_idx],
                truth_number=truth_numbers[truth_idx],
                bi_encoder_score=float(bi_score),
                cross_encoder_score=0.0,  # Not used
                combined_score=float(bi_score),
                match_confidence=self._get_confidence_level(float(bi_score))
            )
            matches.append(match)
            used_model_indices.add(model_idx)
            used_truth_indices.add(truth_idx)
        
        return matches


# Create evaluators for comparison
bi_only_matcher = BiEncoderOnlyMatcher(
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,  # Not actually used
    similarity_threshold=0.5
)

# Create evaluator using bi-encoder only
class BiEncoderOnlyEvaluator(SentenceTransformerNumericalEvaluator):
    def __init__(self, bi_encoder, **kwargs):
        super().__init__(bi_encoder=bi_encoder, cross_encoder=cross_encoder, **kwargs)
        self.matcher = BiEncoderOnlyMatcher(
            bi_encoder=bi_encoder,
            cross_encoder=cross_encoder,
            similarity_threshold=kwargs.get('min_semantic_score', 0.5)
        )

bi_only_evaluator = BiEncoderOnlyEvaluator(
    bi_encoder=bi_encoder,
    relative_tolerance=0.10,
    absolute_tolerance=0.01
)

# Compare results
print("📊 Comparing Bi-Encoder Only vs Bi-Encoder + Cross-Encoder")
print("="*80)

comparison_data = []
for ex in examples:
    # Bi-encoder only
    bi_result = bi_only_evaluator.evaluate(
        question=ex["question"],
        model_answer=ex["model_answer"],
        ground_truth=ex["ground_truth"]
    )
    
    # Combined (already computed)
    combined_result = next(r["result"] for r in results if r["example"]["id"] == ex["id"])
    
    comparison_data.append({
        "Example": ex["name"][:25],
        "Bi-Only Decision": bi_result.overall_decision.value,
        "Combined Decision": combined_result.overall_decision.value,
        "Bi-Only Score": bi_result.overall_score,
        "Combined Score": combined_result.overall_score,
        "Bi-Only Matches": bi_result.matched_pairs,
        "Combined Matches": combined_result.matched_pairs
    })
    
    print(f"\n{ex['name'][:30]}:")
    print(f"  Bi-Encoder Only:     {bi_result.overall_decision.value:10s} (Score: {bi_result.overall_score:.3f})")
    print(f"  Bi + Cross-Encoder:  {combined_result.overall_decision.value:10s} (Score: {combined_result.overall_score:.3f})")

comparison_df = pd.DataFrame(comparison_data)
print("\n\n📋 Comparison Table:")
display(comparison_df)


## 10. Similarity Matrix Visualization

Visualize the semantic similarity between number contexts from model answer and ground truth.


## 11. Configuration Exploration

Explore how different model configurations and thresholds affect evaluation results.


In [ ]:
# Explore different bi-encoder/cross-encoder weight combinations
weight_configs = [
    (1.0, 0.0),   # Bi-encoder only
    (0.7, 0.3),   # Mostly bi-encoder
    (0.5, 0.5),   # Equal weights
    (0.3, 0.7),   # Mostly cross-encoder (default)
    (0.0, 1.0),   # Cross-encoder only
]

print("📊 Effect of Bi-Encoder / Cross-Encoder Weight Configuration")
print("="*80)

config_results = []

for bi_w, ce_w in weight_configs:
    # Create matcher with specific weights
    test_matcher = SemanticNumberMatcher(
        bi_encoder=bi_encoder,
        cross_encoder=cross_encoder,
        bi_encoder_weight=bi_w,
        cross_encoder_weight=ce_w,
        similarity_threshold=0.5
    )
    
    total_matches = 0
    avg_scores = []
    
    for ex in examples:
        model_nums = extractor.extract(ex["model_answer"])
        truth_nums = extractor.extract(ex["ground_truth"])
        matches = test_matcher.match(model_nums, truth_nums)
        total_matches += len(matches)
        avg_scores.extend([m.combined_score for m in matches])
    
    config_results.append({
        'Bi-Encoder Weight': bi_w,
        'Cross-Encoder Weight': ce_w,
        'Config': f"Bi:{bi_w:.1f}/CE:{ce_w:.1f}",
        'Total Matches': total_matches,
        'Avg Combined Score': np.mean(avg_scores) if avg_scores else 0
    })
    
    print(f"  Bi:{bi_w:.1f} / CE:{ce_w:.1f}  →  Matches: {total_matches}, Avg Score: {np.mean(avg_scores) if avg_scores else 0:.3f}")

config_df = pd.DataFrame(config_results)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(config_df))
bars = ax.bar(x, config_df['Avg Combined Score'], color='#3498db', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(config_df['Config'])
ax.set_ylabel('Average Combined Score')
ax.set_xlabel('Weight Configuration')
ax.set_title('Impact of Bi-Encoder/Cross-Encoder Weight on Match Scores')
ax.set_ylim(0, 1)

for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
           f'{config_df.iloc[i]["Total Matches"]} matches', 
           ha='center', fontsize=9)

plt.tight_layout()
plt.show()


## 12. Alternative Sentence Transformer Models

Different models have different strengths. Here are some recommended alternatives:


In [ ]:
# Alternative models to consider
alternative_models = {
    'bi_encoders': {
        'all-MiniLM-L6-v2': {
            'description': 'Fast and good general-purpose model (used in this demo)',
            'size': '~80MB',
            'speed': 'Fast',
            'quality': 'Good'
        },
        'all-mpnet-base-v2': {
            'description': 'Higher quality embeddings, larger model',
            'size': '~420MB',
            'speed': 'Medium',
            'quality': 'Excellent'
        },
        'paraphrase-MiniLM-L6-v2': {
            'description': 'Optimized for paraphrase detection',
            'size': '~80MB',
            'speed': 'Fast',
            'quality': 'Good for similarity'
        },
        'multi-qa-MiniLM-L6-cos-v1': {
            'description': 'Optimized for semantic search and QA',
            'size': '~80MB',
            'speed': 'Fast',
            'quality': 'Good for QA tasks'
        },
        'sentence-t5-base': {
            'description': 'T5-based embeddings, very high quality',
            'size': '~890MB',
            'speed': 'Slow',
            'quality': 'Excellent'
        }
    },
    'cross_encoders': {
        'cross-encoder/ms-marco-MiniLM-L-6-v2': {
            'description': 'Fast passage ranking (used in this demo)',
            'size': '~80MB',
            'speed': 'Fast',
            'quality': 'Good'
        },
        'cross-encoder/ms-marco-MiniLM-L-12-v2': {
            'description': 'Better quality passage ranking',
            'size': '~120MB',
            'speed': 'Medium',
            'quality': 'Very Good'
        },
        'cross-encoder/stsb-roberta-large': {
            'description': 'High quality semantic similarity',
            'size': '~1.3GB',
            'speed': 'Slow',
            'quality': 'Excellent'
        },
        'cross-encoder/nli-deberta-v3-base': {
            'description': 'NLI-based, good for entailment checking',
            'size': '~540MB',
            'speed': 'Medium',
            'quality': 'Excellent for NLI'
        }
    }
}

print("📚 Recommended Sentence Transformer Models")
print("="*80)

print("\n🔵 BI-ENCODERS (for embedding and similarity search):")
print("-"*60)
for name, info in alternative_models['bi_encoders'].items():
    print(f"\n  {name}")
    print(f"    {info['description']}")
    print(f"    Size: {info['size']} | Speed: {info['speed']} | Quality: {info['quality']}")

print("\n\n🔴 CROSS-ENCODERS (for pairwise scoring):")
print("-"*60)
for name, info in alternative_models['cross_encoders'].items():
    print(f"\n  {name}")
    print(f"    {info['description']}")
    print(f"    Size: {info['size']} | Speed: {info['speed']} | Quality: {info['quality']}")


## 13. Production Pipeline

A complete pipeline class for production use.


In [ ]:
class NumericalAnswerEvaluationPipeline:
    """
    Production-ready pipeline for evaluating numerical answers using Sentence Transformers.
    
    Features:
    - Configurable bi-encoder and cross-encoder models
    - Adjustable numerical tolerances
    - Detailed reporting and logging
    - Batch evaluation support
    """
    
    def __init__(
        self,
        bi_encoder_model: str = "all-MiniLM-L6-v2",
        cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        bi_encoder_weight: float = 0.3,
        cross_encoder_weight: float = 0.7,
        relative_tolerance: float = 0.10,
        absolute_tolerance: float = 0.01,
        similarity_threshold: float = 0.5,
        device: str = None
    ):
        """
        Initialize the evaluation pipeline.
        
        Args:
            bi_encoder_model: Name of the bi-encoder model
            cross_encoder_model: Name of the cross-encoder model
            bi_encoder_weight: Weight for bi-encoder score in combined matching
            cross_encoder_weight: Weight for cross-encoder score
            relative_tolerance: Relative error tolerance for numerical comparison
            absolute_tolerance: Absolute error tolerance
            similarity_threshold: Minimum similarity for matching
            device: Device to run models on ('cuda', 'cpu', or None for auto)
        """
        self.config = {
            'bi_encoder_model': bi_encoder_model,
            'cross_encoder_model': cross_encoder_model,
            'bi_encoder_weight': bi_encoder_weight,
            'cross_encoder_weight': cross_encoder_weight,
            'relative_tolerance': relative_tolerance,
            'absolute_tolerance': absolute_tolerance,
            'similarity_threshold': similarity_threshold
        }
        
        # Load models
        print(f"Loading models...")
        self.bi_encoder = SentenceTransformer(bi_encoder_model, device=device)
        self.cross_encoder = CrossEncoder(cross_encoder_model, device=device)
        
        # Initialize evaluator
        self.evaluator = SentenceTransformerNumericalEvaluator(
            bi_encoder=self.bi_encoder,
            cross_encoder=self.cross_encoder,
            relative_tolerance=relative_tolerance,
            absolute_tolerance=absolute_tolerance,
            min_semantic_score=similarity_threshold
        )
        
        # Update matcher weights
        self.evaluator.matcher.bi_encoder_weight = bi_encoder_weight
        self.evaluator.matcher.cross_encoder_weight = cross_encoder_weight
        
        print("Pipeline ready!")
    
    def evaluate_single(
        self, 
        question: str, 
        model_answer: str, 
        ground_truth: str
    ) -> Dict[str, Any]:
        """Evaluate a single answer and return structured results."""
        result = self.evaluator.evaluate(question, model_answer, ground_truth)
        
        return {
            'decision': result.overall_decision.value,
            'overall_score': round(result.overall_score, 4),
            'numerical_accuracy': round(result.numerical_accuracy_score, 4),
            'matching_coverage': round(result.matching_coverage, 4),
            'semantic_score': round(result.semantic_match_score, 4),
            'stats': {
                'model_numbers': result.total_model_numbers,
                'truth_numbers': result.total_truth_numbers,
                'matched_pairs': result.matched_pairs,
                'within_tolerance': result.within_tolerance_count
            },
            'matches': [
                {
                    'model_value': comp.model_value,
                    'truth_value': comp.truth_value,
                    'relative_error': round(comp.relative_error, 4),
                    'bi_encoder_score': round(match.bi_encoder_score, 4),
                    'cross_encoder_score': round(match.cross_encoder_score, 4),
                    'decision': comp.decision.value
                }
                for match, comp in zip(result.matches, result.comparisons)
            ]
        }
    
    def evaluate_batch(
        self, 
        data: List[Dict[str, str]]
    ) -> pd.DataFrame:
        """
        Evaluate a batch of answers.
        
        Args:
            data: List of dicts with 'question', 'model_answer', 'ground_truth' keys
            
        Returns:
            DataFrame with evaluation results
        """
        results = []
        
        for item in data:
            result = self.evaluate_single(
                question=item['question'],
                model_answer=item['model_answer'],
                ground_truth=item['ground_truth']
            )
            result['id'] = item.get('id', len(results))
            results.append(result)
        
        # Create summary DataFrame
        summary = pd.DataFrame([
            {
                'id': r['id'],
                'decision': r['decision'],
                'overall_score': r['overall_score'],
                'numerical_accuracy': r['numerical_accuracy'],
                'matching_coverage': r['matching_coverage'],
                'matched_pairs': r['stats']['matched_pairs'],
                'within_tolerance': r['stats']['within_tolerance']
            }
            for r in results
        ])
        
        return summary
    
    def get_config(self) -> Dict:
        """Return current configuration."""
        return self.config.copy()


# Example usage
print("🚀 Production Pipeline Demo")
print("="*80)

pipeline = NumericalAnswerEvaluationPipeline(
    bi_encoder_model="all-MiniLM-L6-v2",
    cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    bi_encoder_weight=0.3,
    cross_encoder_weight=0.7,
    relative_tolerance=0.10
)

# Single evaluation
single_result = pipeline.evaluate_single(
    question="What is Tesla's revenue?",
    model_answer="Tesla reported revenue of $24.9 billion in Q3 2023, with automotive revenue of $19.6 billion.",
    ground_truth="Tesla's Q3 2023 total revenue was $23.4 billion, including $19.6 billion from automotive."
)

print("\n📊 Single Evaluation Result:")
print(f"  Decision: {single_result['decision']}")
print(f"  Overall Score: {single_result['overall_score']}")
print(f"  Numerical Accuracy: {single_result['numerical_accuracy']}")
print(f"  Matched: {single_result['stats']['matched_pairs']} pairs")

# Batch evaluation
batch_data = [{'id': ex['id'], **ex} for ex in examples[:3]]
batch_results = pipeline.evaluate_batch(batch_data)

print("\n📋 Batch Evaluation Results:")
display(batch_results)


## 14. Best Practices and Recommendations


## 15. Summary

This notebook demonstrated numerical answer evaluation using **Sentence Transformers**:

### Key Components:

1. **NumberContextExtractor**: Extracts numbers with surrounding context from text
2. **SemanticNumberMatcher**: Two-stage matching using bi-encoder retrieval + cross-encoder re-ranking
3. **SentenceTransformerNumericalEvaluator**: Full evaluation pipeline with statistical comparison

### Two-Stage Matching Process:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Model Answer Numbers    ──►  Bi-Encoder  ──►  Similarity Matrix       │
│                                    │                                    │
│  Ground Truth Numbers    ──►  Bi-Encoder  ──►  Candidate Pairs         │
│                                                      │                  │
│                                              Cross-Encoder              │
│                                                      │                  │
│                                              Final Matches ──►  Stats   │
└─────────────────────────────────────────────────────────────────────────┘
```

### Advantages of Sentence Transformer Approach:

- **Deterministic**: Same inputs always produce same outputs
- **Fast**: Can process hundreds of evaluations per second
- **Local**: No API costs, no data privacy concerns
- **Configurable**: Fine-tune weights and thresholds for your domain

### When to Use:

- High-throughput automated evaluation pipelines
- Applications requiring consistent, reproducible scores
- Local/offline evaluation scenarios
- Cost-sensitive deployments

### Compared to LLM-as-Judge:

| Aspect | Sentence Transformers | LLM-as-Judge |
|--------|----------------------|--------------|
| Speed | Very Fast | Slow |
| Cost | Free (local) | API costs |
| Consistency | Deterministic | Variable |
| Context Understanding | Pattern-based | Nuanced |
| Explainability | Scores only | Can explain reasoning |


In [ ]:
best_practices = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║                    BEST PRACTICES FOR NUMERICAL ANSWER EVALUATION               ║
║                           USING SENTENCE TRANSFORMERS                           ║
╚══════════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 1. MODEL SELECTION                                                              │
├─────────────────────────────────────────────────────────────────────────────────┤
│ BI-ENCODER:                                                                     │
│   • Use 'all-MiniLM-L6-v2' for fast, general-purpose matching                  │
│   • Use 'all-mpnet-base-v2' when accuracy is more important than speed         │
│   • Use 'multi-qa-MiniLM-L6-cos-v1' for QA-specific tasks                      │
│                                                                                 │
│ CROSS-ENCODER:                                                                  │
│   • Use 'cross-encoder/ms-marco-MiniLM-L-6-v2' for fast re-ranking            │
│   • Use 'cross-encoder/stsb-roberta-large' for highest accuracy                │
│   • Consider NLI-based models for entailment-checking scenarios                │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 2. WEIGHT CONFIGURATION                                                         │
├─────────────────────────────────────────────────────────────────────────────────┤
│ • Default: bi_encoder=0.3, cross_encoder=0.7                                    │
│   - Good balance between speed and accuracy                                     │
│                                                                                 │
│ • For speed-critical applications: bi_encoder=1.0, cross_encoder=0.0           │
│   - Skip cross-encoder, use bi-encoder similarity only                         │
│                                                                                 │
│ • For accuracy-critical applications: bi_encoder=0.0, cross_encoder=1.0        │
│   - Rely entirely on cross-encoder for matching decisions                      │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 3. TOLERANCE SETTINGS BY DOMAIN                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│ FINANCIAL DATA:                                                                 │
│   relative_tolerance = 0.01-0.05 (1-5% error acceptable)                       │
│   absolute_tolerance = 0.01                                                     │
│                                                                                 │
│ SCIENTIFIC DATA:                                                                │
│   relative_tolerance = 0.005-0.02 (0.5-2% error for precision)                 │
│   order_of_magnitude_tolerance = 0.1 (strict magnitude checking)               │
│                                                                                 │
│ GENERAL KNOWLEDGE:                                                              │
│   relative_tolerance = 0.10-0.15 (10-15% error acceptable)                     │
│   Useful for population, approximate statistics, etc.                          │
│                                                                                 │
│ WEATHER/TEMPERATURE:                                                            │
│   relative_tolerance = 0.05                                                     │
│   absolute_tolerance = 1.0-2.0 (1-2 degree variation acceptable)               │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 4. SEMANTIC MATCHING THRESHOLDS                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│ • similarity_threshold = 0.5 (default)                                          │
│   - Good for most use cases                                                     │
│                                                                                 │
│ • similarity_threshold = 0.7 (strict)                                           │
│   - Use when false positives are costly                                         │
│   - May miss some valid matches                                                 │
│                                                                                 │
│ • similarity_threshold = 0.3 (lenient)                                          │
│   - Use when recall is more important than precision                            │
│   - May produce more false positives                                            │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 5. ADVANTAGES VS LLM-AS-JUDGE                                                   │
├─────────────────────────────────────────────────────────────────────────────────┤
│ SENTENCE TRANSFORMER APPROACH:                                                  │
│   ✅ Deterministic and reproducible results                                     │
│   ✅ No API costs or rate limits                                                │
│   ✅ Fast inference (100s of evaluations per second)                            │
│   ✅ Runs locally, no data privacy concerns                                     │
│   ✅ Consistent scoring across evaluations                                      │
│                                                                                 │
│ LLM-AS-JUDGE APPROACH:                                                          │
│   ✅ Better understanding of context and nuance                                 │
│   ✅ Can explain reasoning for decisions                                        │
│   ✅ Handles edge cases more gracefully                                         │
│   ✅ Can consider qualitative aspects                                           │
│                                                                                 │
│ RECOMMENDATION: Use Sentence Transformers for high-throughput, consistent       │
│ numerical evaluation. Use LLM-as-judge for complex, nuanced assessments.        │
└─────────────────────────────────────────────────────────────────────────────────┘
"""

print(best_practices)


In [ ]:
def visualize_similarity_matrix(result: EvaluationResult, example_name: str):
    """Visualize the bi-encoder similarity matrix for an example."""
    if not result.model_numbers or not result.truth_numbers:
        print(f"No numbers to visualize for {example_name}")
        return
    
    # Get embeddings
    model_embeddings = np.array([n.embedding for n in result.model_numbers])
    truth_embeddings = np.array([n.embedding for n in result.truth_numbers])
    
    # Compute similarity matrix
    sim_matrix = cosine_similarity(model_embeddings, truth_embeddings)
    
    # Create labels
    model_labels = [f"M{i+1}: {n.value:,.4g}" for i, n in enumerate(result.model_numbers)]
    truth_labels = [f"T{i+1}: {n.value:,.4g}" for i, n in enumerate(result.truth_numbers)]
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(max(8, len(truth_labels)), max(6, len(model_labels) * 0.6)))
    
    im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Cosine Similarity')
    
    # Set ticks and labels
    ax.set_xticks(range(len(truth_labels)))
    ax.set_yticks(range(len(model_labels)))
    ax.set_xticklabels(truth_labels, rotation=45, ha='right')
    ax.set_yticklabels(model_labels)
    
    # Add text annotations
    for i in range(len(model_labels)):
        for j in range(len(truth_labels)):
            text = ax.text(j, i, f'{sim_matrix[i, j]:.2f}',
                          ha='center', va='center',
                          color='white' if sim_matrix[i, j] > 0.5 else 'black',
                          fontsize=9)
    
    # Mark matched pairs
    for match in result.matches:
        model_idx = result.model_numbers.index(match.model_number)
        truth_idx = result.truth_numbers.index(match.truth_number)
        rect = plt.Rectangle((truth_idx - 0.5, model_idx - 0.5), 1, 1,
                            fill=False, edgecolor='blue', linewidth=3)
        ax.add_patch(rect)
    
    ax.set_xlabel('Ground Truth Numbers')
    ax.set_ylabel('Model Numbers')
    ax.set_title(f'Bi-Encoder Similarity Matrix: {example_name}\n(Blue boxes = final matches)')
    
    plt.tight_layout()
    plt.show()

# Visualize similarity matrices for selected examples
visualize_similarity_matrix(results[0]["result"], results[0]["example"]["name"])
visualize_similarity_matrix(results[3]["result"], results[3]["example"]["name"])
